# Optimización de Modelos Conjunto Soleado por GMM

In [175]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [176]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [ ]:
datos_dia = datos[datos["Cluster GMM"] == "Soleado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
35,2022-09-02 11:00:00,17036.043251,20,0,71,2,2,14,11,Soleado,Nublado,5030.740421,22189.147406
36,2022-09-02 12:00:00,27523.885172,21,0,57,4,2,13,12,Soleado,Nublado,17036.043251,29196.986647
37,2022-09-02 13:00:00,20596.278869,23,0,45,5,2,11,13,Soleado,Soleado,27523.885172,25478.471342
38,2022-09-02 14:00:00,28500.000000,24,3,37,6,2,9,14,Soleado,Lluvioso,20596.278869,29057.585772
64,2022-09-03 16:00:00,23122.803757,25,16,41,6,3,11,16,Soleado,Lluvioso,25500.000000,25500.000000
85,2022-09-04 13:00:00,27000.000000,22,32,58,10,2,13,13,Soleado,Lluvioso,22850.891263,20400.000000
86,2022-09-04 14:00:00,25168.164594,23,25,51,12,1,12,14,Soleado,Lluvioso,27000.000000,21548.984244
87,2022-09-04 15:00:00,24300.000000,23,12,47,10,1,11,15,Soleado,Lluvioso,25168.164594,25500.000000
88,2022-09-04 16:00:00,19686.387416,24,12,46,7,1,12,16,Soleado,Lluvioso,24300.000000,23122.803757
89,2022-09-04 17:00:00,22006.958065,25,12,47,5,1,12,17,Soleado,Lluvioso,19686.387416,25602.778606


In [178]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [179]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,20,0,71,2,2,14,11,5030.740421,22189.147406
36,21,0,57,4,2,13,12,17036.043251,29196.986647
37,23,0,45,5,2,11,13,27523.885172,25478.471342
38,24,3,37,6,2,9,14,20596.278869,29057.585772
64,25,16,41,6,3,11,16,25500.000000,25500.000000
...,...,...,...,...,...,...,...,...,...
18282,26,0,32,2,1,8,17,25386.000000,22664.000000
18283,25,0,33,1,1,8,18,22872.000000,15736.000000
18284,23,0,38,0,1,8,19,15825.000000,1407.000000
18285,22,0,45,0,1,9,20,1450.000000,0.000000


In [180]:
y = datos_dia[['Generación']]
y

,Generación
35,17036.043251
36,27523.885172
37,20596.278869
38,28500.000000
64,23122.803757
...,...
18282,22872.000000
18283,15825.000000
18284,1450.000000
18285,0.000000


Dividimos entrenamiento, validación y prueba

In [181]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [182]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 4873, y_train: 4873
X_val: 1044, y_val: 1044
X_test: 1045, y_test: 1045


## Escalar con MinMaxScaler

In [183]:
from sklearn.preprocessing import MinMaxScaler

In [184]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [185]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.52631579 0.         0.69473684 ... 0.33333333 0.16769135 0.73963825]
 [0.55263158 0.         0.54736842 ... 0.4        0.56786811 0.97323289]
 [0.60526316 0.         0.42105263 ... 0.46666667 0.91746284 0.84928238]
 ...
 [0.21052632 0.         0.48421053 ... 0.06666667 0.         0.        ]
 [0.18421053 0.         0.50526316 ... 0.13333333 0.         0.02536667]
 [0.15789474 0.         0.53684211 ... 0.2        0.01276667 1.        ]]
(4873, 9)


In [186]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,0.526316,0.000000,0.694737,0.142857,0.333333,0.777778,0.333333,0.167691,0.739638
36,0.552632,0.000000,0.547368,0.285714,0.333333,0.722222,0.400000,0.567868,0.973233
37,0.605263,0.000000,0.421053,0.357143,0.333333,0.611111,0.466667,0.917463,0.849282
38,0.631579,0.061224,0.336842,0.428571,0.333333,0.500000,0.533333,0.686543,0.968586
64,0.657895,0.326531,0.378947,0.428571,0.666667,0.611111,0.666667,0.850000,0.850000
...,...,...,...,...,...,...,...,...,...
12570,0.763158,0.000000,0.084211,0.142857,0.666667,0.111111,0.733333,1.000000,1.000000
12583,0.210526,0.000000,0.505263,0.000000,0.000000,0.055556,0.000000,0.000000,0.000000
12584,0.210526,0.000000,0.484211,0.000000,0.000000,0.111111,0.066667,0.000000,0.000000
12585,0.184211,0.000000,0.505263,0.000000,0.333333,0.111111,0.133333,0.000000,0.025367


In [187]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.31578947 0.         0.37894737 ... 0.26666667 0.29363333 1.        ]
 [0.42105263 0.         0.27368421 ... 0.33333333 0.6314     1.        ]
 [0.52631579 0.         0.21052632 ... 0.4        0.67536667 1.        ]
 ...
 [0.92105263 0.         0.14736842 ... 0.6        0.9326     0.97063333]
 [0.94736842 0.         0.11578947 ... 0.66666667 0.97063333 0.95473333]
 [0.97368421 0.         0.10526316 ... 0.73333333 0.95686667 0.8543    ]]
(1044, 9)


In [188]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
12587,0.315789,0.0,0.378947,0.214286,0.333333,0.055556,0.266667,0.293633,1.000000
12588,0.421053,0.0,0.273684,0.285714,0.333333,0.055556,0.333333,0.631400,1.000000
12589,0.526316,0.0,0.210526,0.357143,0.333333,0.000000,0.400000,0.675367,1.000000
12590,0.605263,0.0,0.178947,0.428571,0.333333,0.055556,0.466667,0.656667,1.000000
12591,0.657895,0.0,0.147368,0.357143,0.333333,0.000000,0.533333,0.713833,1.000000
...,...,...,...,...,...,...,...,...,...
15038,0.842105,0.0,0.210526,1.000000,0.000000,0.555556,0.466667,0.936900,0.928933
15039,0.868421,0.0,0.178947,0.857143,0.000000,0.500000,0.533333,0.928933,0.932600
15040,0.921053,0.0,0.147368,0.642857,0.000000,0.444444,0.600000,0.932600,0.970633
15041,0.947368,0.0,0.115789,0.357143,0.000000,0.388889,0.666667,0.970633,0.954733


In [189]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.94736842 0.         0.10526316 ... 0.8        0.85466667 0.71996667]
 [0.92105263 0.         0.11578947 ... 0.86666667 0.72766667 0.2737    ]
 [0.86842105 0.         0.14736842 ... 0.93333333 0.2827     0.0095    ]
 ...
 [0.60526316 0.         0.34736842 ... 0.86666667 0.5275     0.0469    ]
 [0.57894737 0.         0.42105263 ... 0.93333333 0.04833333 0.        ]
 [0.52631579 0.         0.51578947 ... 1.         0.         0.        ]]
(1045, 9)


In [190]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
15043,0.947368,0.0,0.105263,0.142857,0.333333,0.333333,0.800000,0.854667,0.719967
15044,0.921053,0.0,0.115789,0.071429,0.333333,0.277778,0.866667,0.727667,0.273700
15045,0.868421,0.0,0.147368,0.000000,0.333333,0.388889,0.933333,0.282700,0.009500
15046,0.789474,0.0,0.263158,0.000000,0.000000,0.611111,1.000000,0.010167,0.000000
15055,0.500000,0.0,0.852632,0.000000,0.000000,0.888889,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
18282,0.684211,0.0,0.284211,0.142857,0.000000,0.444444,0.733333,0.846200,0.755467
18283,0.657895,0.0,0.294737,0.071429,0.000000,0.444444,0.800000,0.762400,0.524533
18284,0.605263,0.0,0.347368,0.000000,0.000000,0.444444,0.866667,0.527500,0.046900
18285,0.578947,0.0,0.421053,0.000000,0.000000,0.500000,0.933333,0.048333,0.000000


In [191]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [192]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.51282051 0.         0.70103093 ... 0.33333333 0.16769135 0.73963825]
 [0.53846154 0.         0.55670103 ... 0.4        0.56786811 0.97323289]
 [0.58974359 0.         0.43298969 ... 0.46666667 0.91746284 0.84928238]
 ...
 [0.58974359 0.         0.36082474 ... 0.86666667 0.5275     0.0469    ]
 [0.56410256 0.         0.43298969 ... 0.93333333 0.04833333 0.        ]
 [0.51282051 0.         0.5257732  ... 1.         0.         0.        ]]
(6962, 9)


In [193]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,0.512821,0.000000,0.701031,0.142857,0.333333,0.777778,0.333333,0.167691,0.739638
36,0.538462,0.000000,0.556701,0.285714,0.333333,0.722222,0.400000,0.567868,0.973233
37,0.589744,0.000000,0.432990,0.357143,0.333333,0.611111,0.466667,0.917463,0.849282
38,0.615385,0.058824,0.350515,0.428571,0.333333,0.500000,0.533333,0.686543,0.968586
64,0.641026,0.313725,0.391753,0.428571,0.666667,0.611111,0.666667,0.850000,0.850000
...,...,...,...,...,...,...,...,...,...
18282,0.666667,0.000000,0.298969,0.142857,0.000000,0.444444,0.733333,0.846200,0.755467
18283,0.641026,0.000000,0.309278,0.071429,0.000000,0.444444,0.800000,0.762400,0.524533
18284,0.589744,0.000000,0.360825,0.000000,0.000000,0.444444,0.866667,0.527500,0.046900
18285,0.564103,0.000000,0.432990,0.000000,0.000000,0.500000,0.933333,0.048333,0.000000


In [194]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [195]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.56786811]
 [0.91746284]
 [0.68654263]
 ...
 [0.        ]
 [0.01276667]
 [0.29363333]]
(4873, 1)


In [196]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
35,0.567868
36,0.917463
37,0.686543
38,0.950000
64,0.770760
...,...
12570,1.000000
12583,0.000000
12584,0.000000
12585,0.012767


In [197]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.6314    ]
 [0.67536667]
 [0.65666667]
 ...
 [0.97063333]
 [0.95686667]
 [0.85466667]]
(1044, 1)


In [198]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12587,0.631400
12588,0.675367
12589,0.656667
12590,0.713833
12591,0.627167
...,...
15038,0.928933
15039,0.932600
15040,0.970633
15041,0.956867


In [199]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.72766667]
 [0.2827    ]
 [0.01016667]
 ...
 [0.04833333]
 [0.        ]
 [0.        ]]
(1045, 1)


In [200]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15043,0.727667
15044,0.282700
15045,0.010167
15046,0.000000
15055,0.000000
...,...
18282,0.762400
18283,0.527500
18284,0.048333
18285,0.000000


In [201]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [202]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.56786811]
 [0.91746284]
 [0.68654263]
 ...
 [0.04833333]
 [0.        ]
 [0.        ]]
(6962, 1)


In [203]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
35,0.567868
36,0.917463
37,0.686543
38,0.950000
64,0.770760
...,...
18282,0.762400
18283,0.527500
18284,0.048333
18285,0.000000


## Preparación para Redes Neuronales

In [204]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [205]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [206]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (4825, 48, 9), y_train: (4825, 1)
X_val: (996, 48, 9), y_val: (996, 1)
X_test: (997, 48, 9), y_test: (997, 1)


## Optuna

In [207]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [ ]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-12 02:27:09,519] A new study created in memory with name: no-name-b6e1672a-ab53-4b2f-8691-dc8c65f4aa0a
[W 2025-03-12 02:27:09,537] Trial 0 failed with parameters: {'num_leaves': 229, 'subsample': 0.4963396364209546, 'colsample_bytree': 0.5377411730853147, 'min_data_in_leaf': 44} because of the following error: NameError("name 'gpu_support' is not defined").
Traceback (most recent call last):
  File "c:\Users\Claudia\anaconda3\envs\tesina_env\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Claudia\AppData\Local\Temp\ipykernel_20756\1313049950.py", line 22, in objective
    device= gpu_support
NameError: name 'gpu_support' is not defined
[W 2025-03-12 02:27:09,542] Trial 0 failed with value None.


NameError: name 'gpu_support' is not defined

### Random Forest

In [212]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-11 19:25:57,467] A new study created in memory with name: no-name-929e11b1-6e85-48ae-adf7-933b9fc550cb
[I 2025-03-11 19:26:13,962] Trial 0 finished with value: 0.011639761189434815 and parameters: {'n_estimators': 300, 'max_depth': 35, 'min_samples_split': 15, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 0 with value: 0.011639761189434815.
[I 2025-03-11 19:26:24,255] Trial 1 finished with value: 0.009097836418609567 and parameters: {'n_estimators': 250, 'max_depth': 40, 'min_samples_split': 13, 'min_samples_leaf': 8, 'bootstrap': False}. Best is trial 1 with value: 0.009097836418609567.
[I 2025-03-11 19:26:38,958] Trial 2 finished with value: 0.011574890872546045 and parameters: {'n_estimators': 300, 'max_depth': 25, 'min_samples_split': 18, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 1 with value: 0.009097836418609567.
[I 2025-03-11 19:26:59,495] Trial 3 finished with value: 0.012045692846609326 and parameters: {'n_estimators': 400, 'max_depth': 3

Mejores hiperparámetros: {'n_estimators': 400, 'max_depth': 45, 'min_samples_split': 17, 'min_samples_leaf': 10, 'bootstrap': True}


### CTNET

In [213]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    
    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [214]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [215]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-11 19:35:38,750] A new study created in memory with name: no-name-49fbce82-cf33-4436-b2f8-44cb7b02117d
[I 2025-03-11 20:01:56,672] Trial 0 finished with value: 0.11877622455358505 and parameters: {'head_size': 4, 'num_heads': 8, 'ff_dim': 112, 'num_transformer_blocks': 4, 'mlp_units_1': 320, 'mlp_units_2': 160, 'dropout': 0.23262113635016815, 'mlp_dropout': 0.25004517288331674, 'learning_rate': 2.00581114844137e-05, 'batch_size': 512}. Best is trial 0 with value: 0.11877622455358505.
[I 2025-03-11 20:23:06,728] Trial 1 finished with value: 0.04681694507598877 and parameters: {'head_size': 6, 'num_heads': 7, 'ff_dim': 80, 'num_transformer_blocks': 2, 'mlp_units_1': 448, 'mlp_units_2': 256, 'dropout': 0.303020430244923, 'mlp_dropout': 0.2502938239162138, 'learning_rate': 0.00013214377235462986, 'batch_size': 512}. Best is trial 1 with value: 0.04681694507598877.
[I 2025-03-11 20:38:52,645] Trial 2 finished with value: 0.03452784940600395 and parameters: {'head_size': 5, 'num_h

Mejores hiperparámetros: {'head_size': 5, 'num_heads': 5, 'ff_dim': 112, 'num_transformer_blocks': 2, 'mlp_units_1': 192, 'mlp_units_2': 128, 'dropout': 0.2933459854281062, 'mlp_dropout': 0.3522227742199008, 'learning_rate': 0.002180590014884393, 'batch_size': 512}


### Forescasting

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())  
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-12 05:35:50,225] A new study created in memory with name: no-name-7b53bf29-4e75-408e-8534-cb384d67add5
[I 2025-03-12 05:40:51,302] Trial 1 finished with value: 0.10972427576780319 and parameters: {'filters': 64, 'kernel_size': 2, 'lstm_units_1': 256, 'lstm_units_2': 32, 'lstm_units_3': 32, 'dropout_lstm': 0.23069405568410684, 'dropout_dense': 0.1555943975788619, 'learning_rate': 0.00919904200937645, 'batch_size': 128}. Best is trial 1 with value: 0.10972427576780319.
[I 2025-03-12 05:49:22,920] Trial 4 finished with value: 0.1219572126865387 and parameters: {'filters': 64, 'kernel_size': 5, 'lstm_units_1': 256, 'lstm_units_2': 128, 'lstm_units_3': 16, 'dropout_lstm': 0.27869011858723536, 'dropout_dense': 0.21369646828833366, 'learning_rate': 0.007644977299868267, 'batch_size': 512}. Best is trial 1 with value: 0.10972427576780319.
[I 2025-03-12 05:59:51,434] Trial 5 finished with value: 0.17393772304058075 and parameters: {'filters': 32, 'kernel_size': 4, 'lstm_units_1': 128

### Photovoltaic

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)